# Energies for [Co(H2O)5CO2]^3+ (singlet / quintet)

Reproduces paper Figures 2, 3, 4 (gas) and 13, 14, 15 (solvent):
potential energy curves, singlet-quintet gaps and equilibrium energies vs. active-space size,
for ROHF/RHF, HCI, (U)CCSD(T) and SQD at 34/38/46/50 qubits.

Input: `./data/energies_charge3.csv`. Output: `./figures/`.

### Imports

In [ ]:
import matplotlib
#matplotlib.rcParams['text.usetex'] = True
matplotlib.rcParams['text.usetex'] = False
matplotlib.rcParams['mathtext.fontset'] = 'cm'
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick


import numpy as np
import pandas as pd
import scienceplots

### Setup:

In [ ]:
basis_set = 'def2-TZVP'
distance = [1.44, 1.97, 2.51, 3.04, 3.57, 4.10, 4.63, 5.16, 5.69, 6.23]   # Co-O(CO2) distance [Angstrom] for shifts -0.15 ... 1.2
ylim = (-1948.7, -1948.14)

n_qubits_list = [34, 38, 46, 50]
n_elec = { # first medium, then n_qubits, then spin value
    "solvent":{
        34: {0: 22, 4: 22},
        38: {0: 24, 4: 24},
        46: {0: 28, 4: 28},
        50: {0: 30, 4: 30}
    },
    "gas":{
        34: {0: 22, 4: 22},
        38: {0: 24, 4: 24},
        46: {0: 28, 4: 28},
        50: {0: 30, 4: 30}
    }
}

# Energies (Hartree) of all methods, read from ./data/energies_charge3.csv
energies = pd.read_csv('./data/energies_charge3.csv')

def results_dict(medium):
    """Rebuild {'<METHOD>_<State>[_<N>so]': [E(shift) ...]} for one medium."""
    out = {}
    sub = energies[energies['medium'] == medium]
    for (method, state, nq), grp in sub.groupby(['method', 'state', 'n_qubits'], dropna=False, sort=False):
        key = f'{method}_{state.capitalize()}' + ('' if pd.isna(nq) else f'_{int(nq)}so')
        out[key] = grp.sort_values('distance_angstrom')['energy_hartree'].tolist()
    return out

gas_results = results_dict('gas')
df_gas = pd.DataFrame(gas_results, index=distance)

solvent_results = results_dict('solvent')
df_solvent = pd.DataFrame(solvent_results, index=distance)

### Helpers

In [ ]:
def plot_energy_differences(df, distance, n_qubits_list, n_elec, medium, if_HCI=True):
    df_diff = pd.DataFrame(index=distance, columns=None)

    #with plt.style.context(['science', 'no-latex']):
    fig,axs = plt.subplots(1, 1, figsize=(6.5, 5))
    l0, = plt.plot(distance, np.abs(df["ROHF_Quintet"] - df["ROHF_Singlet"]), color='gray',linestyle='-',marker='X', markersize=7, label='RHF / ROHF',alpha=0.7)
    
    df_diff['ROHF'] = np.abs(df["ROHF_Quintet"] - df["ROHF_Singlet"])

    colormap = plt.get_cmap('tab10')
    n_colors = len(n_qubits_list)
    colors = ['C3','C1','C2','C0']##colormap(np.linspace(0.2, 1, n_colors))
    
    l1 = None
    if if_HCI:
        l1, = plt.plot([None], [None], ':o', color='black', label=f'HCI')
    l2,  = plt.plot([None], [None], '--', color='black', label=f'(U)CCSD(T)')
    l3,  = plt.plot([None], [None], '-v', color='black', label=f'SQD')
    
    # add the legend immediately
    legend1 = axs.legend(frameon=True, 
                        handles=[l0, l1, l2, l3] if if_HCI else [l0, l2, l3],ncols=2,
                        loc='lower left' if medium=='solvent' else 'upper left')
    as_handles = []
    for n_so in n_qubits_list:
        l, = plt.plot([None], 
                    [None], 
                    "-", 
                    color=colors[n_qubits_list.index(n_so)],
                    label=f'{n_so}-qubits ({n_elec[medium][n_so][0]}e, {n_so//2}o)',alpha=0.7)

        as_handles.append(l)
        if if_HCI:
            plt.plot(distance, 
                        np.abs(df[f'HCI_Quintet_{n_so}so'] - df[f'HCI_Singlet_{n_so}so']), 
                        ':o', 
                        color=colors[n_qubits_list.index(n_so)],
                        markersize=7, label=f'HCI {n_so}-nso',alpha=0.7)
            df_diff[f'HCI_{n_so}so'] = np.abs(df[f'HCI_Quintet_{n_so}so'] - df[f'HCI_Singlet_{n_so}so'])
        
        for method in ['CCSD(T)']:
            plt.plot(distance, 
                    np.abs(df[f'{method}_Quintet_{n_so}so'] - df[f'{method}_Singlet_{n_so}so']),
                    '--',
                    color=colors[n_qubits_list.index(n_so)],
                    label=None,alpha=0.7)
            df_diff[f'{method}_{n_so}so'] = np.abs(df[f'{method}_Quintet_{n_so}so'] - df[f'{method}_Singlet_{n_so}so'])
    
        plt.plot(distance, 
                np.abs(df[f'SQD_Quintet_{n_so}so'] - df[f'SQD_Singlet_{n_so}so']), 
                '-v', 
                color=colors[n_qubits_list.index(n_so)],
                markersize=7, label=None,alpha=0.7)
        df_diff[f'SQD_{n_so}so'] = np.abs(df[f'SQD_Quintet_{n_so}so'] - df[f'SQD_Singlet_{n_so}so'])
    
    axs.add_artist(legend1)
    legend2 = axs.legend(frameon=True, 
                        loc='lower right',
                        handles=as_handles
                        #    bbox_to_anchor=(1, -0.1)
                        )

    # axs.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.1f'))
    plt.grid()
    plt.ylabel('$E_{\\text{singlet}}-E_{\\text{quintet}}$ [Ha]', fontsize=14)
    plt.xlabel('Distance [$\mathrm{\AA}$]', fontsize=14)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.title(f"Energy Difference for $[Co(H_2O)_5CO_2]^{{3+}}$ ({medium})",fontsize=14)
    plt.savefig(f"./figures/{medium}_charge_3_diff_results.pdf")
    plt.show()
    
    return df_diff


def plot_equilibrium_point(df, n_qubits_list, medium, if_hci=True):
    row = df.iloc[1]  # equilibrium geometry

    # matplotlib default cycle per method; gray for ROHF
    c_sqd, c_hci, c_ccsdt, c_rohf = 'C0', 'C1', 'C2', 'black'

    # dotted line = Singlet, solid line = Quintet; marker per method
    # with plt.style.context(['science', 'no-latex']):
    #fig, axs = plt.subplots(1, 1, figsize=(7, 5))#, dpi=300)

    plt.figure(figsize=[6.5, 5])

    # ROHF: flat reference lines (gray)
    plt.axhline(row['ROHF_Singlet'], linestyle=':', color=c_rohf)
    plt.axhline(row['ROHF_Quintet'], linestyle='-', color=c_rohf)

    # (U)CCSD(T) (C2, triangle marker)
    plt.plot(n_qubits_list, [row[f'CCSD(T)_Singlet_{n}so'] for n in n_qubits_list], ':s',markersize=8, color=c_ccsdt)
    plt.plot(n_qubits_list, [row[f'CCSD(T)_Quintet_{n}so'] for n in n_qubits_list], '-s',markersize=8,color=c_ccsdt)

    # HCI (C1, square marker)
    if if_hci:
        plt.plot(n_qubits_list, [row[f'HCI_Singlet_{n}so'] for n in n_qubits_list], ':o',markersize=8, color=c_hci,alpha=0.8)
        plt.plot(n_qubits_list, [row[f'HCI_Quintet_{n}so'] for n in n_qubits_list], '-o',markersize=8, color=c_hci,alpha=0.8)

    # SQD (C0, circle marker)
    plt.plot(n_qubits_list, [row[f'SQD_Singlet_{n}so'] for n in n_qubits_list], ':^',markersize=8, color=c_sqd,alpha=0.8)
    plt.plot(n_qubits_list, [row[f'SQD_Quintet_{n}so'] for n in n_qubits_list], '-^',markersize=8, color=c_sqd,alpha=0.8)

    # legend (dummy handles)
    plt.plot([], [], ':', color='gray', label='Singlet')
    plt.plot([], [], '-', color='gray', label='Quintet')
    plt.plot([], [], '-', color=c_rohf, label='RHF / ROHF')
    plt.plot([], [], '^', color=c_sqd, label='SQD')
    if if_hci:
        plt.plot([], [], 'o', color=c_hci, label='HCI')
    plt.plot([], [], 's', color=c_ccsdt, label='(U)CCSD(T)')

    plt.legend(frameon=True, ncols=2, loc='upper right',fontsize=12)
    plt.grid()
    if medium == 'gas':
        plt.ylim(-1948.6, -1948.28)
    #axs.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.2f'))
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.xlabel('Number of Qubits', fontsize=14)
    plt.ylabel('Energy [Ha]', fontsize=14)
    plt.title(f"$[Co(H_2O)_5CO_2]^{{3+}}$ ({medium})", fontsize=14)
    plt.tight_layout()
    plt.savefig(f"./figures/{medium}_charge_3_eq_point_results.pdf")#, dpi=300)
    plt.show()
        
        
def plot_PES(df, n_qubits_list, n_elec, medium):
    #with plt.style.context(['science', 'no-latex']):
    fig,axs = plt.subplots(2, 4, figsize=(14, 6))#, dpi=300) # 250
    for i, n_so in enumerate(n_qubits_list):
        for j, (spin_state, spin_number) in enumerate([('Quintet', 4), ('Singlet', 0)]):
            n_e = (n_elec[medium][n_so][spin_number]//2 + spin_number//2,
                n_elec[medium][n_so][spin_number]//2 - spin_number//2)

            axs[j][i].plot(df[f"ROHF_{spin_state}"], '-o', label='RHF / ROHF')
            axs[j][i].plot(df[f"HCI_{spin_state}_{n_so}so"], '-o', label=f'HCI'# ($\epsilon_1$={e1_det:.0e}, $\epsilon_2$={e2_det})'
                        )            
            axs[j][i].plot(df[f"SQD_{spin_state}_{n_so}so"], '-v', label=f'SQD')
            axs[j][i].plot(df[f"CCSD(T)_{spin_state}_{n_so}so"], label='(U)CCSD(T)')    
            
            axs[1][i].legend(frameon=True, 
                    loc='lower right'
                    # bbox_to_anchor=(1, -0.15)
                    )
            if medium=='gas':
                axs[j][i].set_ylim(*ylim)
            axs[j][i].yaxis.set_major_formatter(mtick.FormatStrFormatter('%.2f'))

            axs[j][i].grid()
            
            axs[j][0].set_ylabel(f'E ({spin_state}) [Ha]',fontsize=14)
            #axs[j][0].set_ylabel(f'Energy [Ha]')
            axs[1][i].set_xlabel('Distance [$\mathrm{\AA}$]',fontsize=14)
            axs[0][i].set_title(f"({n_elec[medium][n_so][spin_number]}e, {n_so//2}o) - {n_so}-qubits",fontsize=14)
        
    plt.suptitle(f"Quintet and Singlet Energies for $[Co(H_2O)_5CO_2]^{{3+}}$ ({medium})", fontsize=16)
    plt.tight_layout()
    plt.savefig(f"./figures/{medium}_charge_3_results.pdf")#, dpi=300)
    plt.show()

In [ ]:
medium = 'gas'

plot_PES(df_gas, n_qubits_list, n_elec, medium)
df_diff = plot_energy_differences(df_gas, distance, n_qubits_list, n_elec, medium)
plot_equilibrium_point(df_gas, n_qubits_list, medium)

In [ ]:
medium = 'solvent'

plot_PES(df_solvent, n_qubits_list, n_elec, medium)
df_diff = plot_energy_differences(df_solvent, distance, n_qubits_list, n_elec, medium)
plot_equilibrium_point(df_solvent, n_qubits_list, medium)